# Continual Counsel — Colab Training Session

This notebook contains **bootstrap cells only**. No pipeline logic lives here.
All training code is in `src/offline/` of the cloned repository.

If the runtime dies, re-run all cells from the top. The pipeline is idempotent
with respect to partially-completed runs (it writes to timestamped directories).

**Before running:** set `REPO_URL`, `COMMIT_OR_TAG`, and `REGIME_NAME` in Cell 1.

In [ ]:
# ── Cell 1: Clone the repo at a pinned commit ─────────────────────────────────
# Pin the commit explicitly. The export bundle manifest records this hash so
# every locally-registered adapter can be traced to the exact code that produced it.
REPO_URL = "https://github.com/YOUR_ORG/continual-counsel.git"  # <-- change this
COMMIT_OR_TAG = "main"   # <-- pin to a specific commit SHA or tag for reproducibility
REGIME_NAME = "regime_q1"  # <-- the regime being updated this session

import subprocess, sys

result = subprocess.run(
    f"git clone {REPO_URL} continual-counsel && "
    f"cd continual-counsel && git checkout {COMMIT_OR_TAG}",
    shell=True, check=True, text=True, capture_output=True
)
print(result.stdout or result.stderr)

commit = subprocess.check_output(
    "git -C continual-counsel rev-parse HEAD", shell=True, text=True
).strip()
print(f"Cloned at commit: {commit}")

%cd continual-counsel

In [ ]:
# ── Cell 2: Install offline dependencies ─────────────────────────────────────
# Never installs requirements-online.txt — the Colab environment has no reason
# for faiss-cpu's CPU wheel build or llama-cpp-python's CPU wheel.
!pip install -r requirements-offline.txt -q

In [ ]:
# ── Cell 3: Fetch the input bundle ───────────────────────────────────────────
# Mount Google Drive (or modify this cell to pull from wherever your input bundle lives).
# The input bundle contains:
#   - data/regimes/<regime>/  (new regulatory docs for this quarter)
#   - faiss_snapshot/          (current FAISS index, needed for Step A grounding filter)
#   - adapter_stack/           (current adapter or merged adapter + basis checkpoint)
#
# These are NOT committed to git: regulatory corpora and adapter weights have no
# business in version-control history.

INPUT_BUNDLE_DIR = "/content/input_bundle"  # adjust to where you placed the bundle

# Example: mount Drive and point to the bundle folder there
# from google.colab import drive
# drive.mount('/content/drive')
# INPUT_BUNDLE_DIR = '/content/drive/MyDrive/continual_counsel/input_bundles/regime_q1'

import os
if not os.path.isdir(INPUT_BUNDLE_DIR):
    print(f"WARNING: Input bundle directory not found: {INPUT_BUNDLE_DIR}")
    print("For the toy 2-regime demo, the data/regimes/ files are already in the repo.")
    print("The toy run will use repo data directly; set INPUT_BUNDLE_DIR for real use.")
    INPUT_BUNDLE_DIR = None  # pipeline.py handles None by using repo-local data
else:
    print(f"Input bundle found: {INPUT_BUNDLE_DIR}")

In [ ]:
# ── Cell 4: Run the pipeline ──────────────────────────────────────────────────
# All logic is in src/offline/pipeline.py. This cell is a shell call, not logic.

import subprocess, sys

cmd = ["bash", "scripts/run_quarter_update_colab.sh", REGIME_NAME]
if INPUT_BUNDLE_DIR:
    cmd += ["--input-bundle", INPUT_BUNDLE_DIR]

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"Pipeline failed with exit code {proc.returncode}")
print("Pipeline complete.")

In [ ]:
# ── Cell 5: Assemble and retrieve the export bundle ───────────────────────────
# Produces exports/<regime>_<timestamp>.tar.gz with a complete manifest.
# Transfer to local machine via Drive, manual download, or scp over a tunnel.
# The transfer mechanism is not hardcoded — use whatever works for your setup.

# Optional: set DRIVE_DEST to copy directly to Drive
DRIVE_DEST = ""  # e.g. "/content/drive/MyDrive/continual_counsel/exports"

cmd = ["bash", "scripts/assemble_export_bundle.sh", REGIME_NAME]
if DRIVE_DEST:
    cmd.append(DRIVE_DEST)

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"Bundle assembly failed with exit code {proc.returncode}")

# List what was produced
import os, glob
archives = sorted(glob.glob(f"exports/{REGIME_NAME}_*.tar.gz"))
if archives:
    latest = archives[-1]
    size_mb = os.path.getsize(latest) / 1024**2
    print(f"\nExport bundle: {latest} ({size_mb:.1f} MB)")
    if not DRIVE_DEST:
        print("Download via: Files panel (left sidebar) > navigate to exports/")
        from google.colab import files
        # Uncomment to trigger browser download:
        # files.download(latest)